# Chương 6: Hệ đa Agent – State, Message, hợp hàm và kiểm tra lỗi

    Hệ điều phối xác định với bốn vai trò; chưa phải nhóm LLM tự lập kế hoạch.
    Chạy trên corpus và FLAN-T5-small cục bộ. Mỗi cell đi kèm điều cần giải thích.

In [1]:
# Cell AGENT-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

Project: E:\WINDOW\Project\intelligent-system-assignment-01


## 1. Mã thực của State, Message và các tác tử

    State chứa câu hỏi, evidence, graph_context, answer, status và messages.
    Message ghi sender, receiver và payload. Bốn phép chuyển trạng thái hợp thành
    S4 = Verifier ∘ Writer ∘ Graph ∘ Retrieval (S0).
    Cell tiếp theo định nghĩa đầy đủ các lớp, không chỉ import tên.

In [2]:
# Cell AGENT-02
"""Bounded, role-based multi-agent demonstration with explicit messages."""
from dataclasses import dataclass, field


@dataclass
class Message:
    sender: str
    receiver: str
    payload: dict


@dataclass
class State:
    question: str
    domain: str
    report: str | None = None
    evidence: list = field(default_factory=list)
    graph_context: list = field(default_factory=list)
    answer: str = ''
    status: str = 'pending'
    messages: list = field(default_factory=list)

    def send(self, sender, receiver, **payload):
        self.messages.append(Message(sender, receiver, payload))


class RetrievalAgent:
    def __init__(self, engine):
        self.engine = engine

    def act(self, state):
        state.evidence = self.engine.retrieve(
            state.question, state.domain, k=3, report=state.report)
        state.send('retrieval', 'graph',
                   ids=[e['id'] for e in state.evidence])


class GraphAgent:
    def __init__(self, graphs):
        self.graphs = graphs

    def act(self, state):
        graph = self.graphs.get(state.domain)
        if graph is not None:
            for item in state.evidence:
                node = 'passage:' + item['id']
                if node in graph:
                    state.graph_context.append({
                        'id': item['id'],
                        'parents': list(graph.predecessors(node))})
        state.send('graph', 'writer',
                   matched=len(state.graph_context))


class WriterAgent:
    def __init__(self, generator):
        self.generator = generator

    def act(self, state):
        if not state.evidence:
            state.status = 'insufficient_evidence'
        else:
            state.answer = self.generator.generate(
                state.question, state.evidence)
        state.send('writer', 'verifier', answer=state.answer)


class VerifierAgent:
    def __init__(self, corpora):
        self.allowed = {d: {x['id'] for x in c}
                        for d, c in corpora.items()}

    def act(self, state):
        ids = [e['id'] for e in state.evidence]
        valid = bool(ids) and all(
            cid in self.allowed[state.domain] for cid in ids)
        if not valid or not state.answer.strip():
            state.answer = 'Insufficient evidence'
            state.status = 'insufficient_evidence'
        else:
            state.status = 'generated_unverified'
        state.send('verifier', 'coordinator', citation_ids_valid=valid,
                   semantic_support_checked=False, status=state.status)


class MultiAgentTeam:
    def __init__(self, engine, graphs):
        self.agents = [RetrievalAgent(engine), GraphAgent(graphs),
                       WriterAgent(engine.generator),
                       VerifierAgent(engine.corpora)]

    def ask(self, question, domain, report=None):
        if domain not in ('diabetes', 'finance'):
            raise ValueError('Unsupported domain')
        if not question.strip():
            raise ValueError('Empty question')
        state = State(question, domain, report)
        for agent in self.agents:  # Four bounded turns; no retry loop.
            agent.act(state)
        return state

## 2. Khởi tạo phụ thuộc

    `rag()` nạp corpus, hai bộ truy hồi và generator; `load_graph` nạp đồ thị
    JSON nén; MultiAgentTeam gắn từng phụ thuộc cho vai trò cần dùng.
    Chỉ dùng một generator chung, không gọi bốn mô hình độc lập.

In [3]:
# Cell AGENT-03
from deployment.chapter04.app import rag, agent
from src.chapter04.knowledge import load_graph
graphs = {'finance':load_graph(
    ROOT/'knowledge_graph/chapter04/finance_graph.json.gz')}
team = MultiAgentTeam(rag(),graphs)
print([type(a).__name__ for a in team.agents])

['RetrievalAgent', 'GraphAgent', 'WriterAgent', 'VerifierAgent']


## 3. Quan sát hợp hàm bằng cách chạy riêng từng tác tử

    Sau mỗi act, in số bằng chứng, ngữ cảnh đồ thị, câu trả lời, status và
    thông điệp vừa tạo. Retrieval không viết đáp án; Graph bổ sung nguồn cha;
    Writer sinh; Verifier chỉ kiểm tra cấu trúc nguồn, chưa xác minh ngữ nghĩa.

In [4]:
# Cell AGENT-04
state = State('What is the percentage change in revenue?','finance')
trace = []
for worker in team.agents:
    worker.act(state)
    trace.append({'agent':type(worker).__name__,
        'evidence':len(state.evidence),'graph':len(state.graph_context),
        'answer':state.answer,'status':state.status})
    message = state.messages[-1]
    print(message.sender,'->',message.receiver,message.payload)
display(pd.DataFrame(trace))
assert len(state.messages)==4
assert not state.messages[-1].payload['semantic_support_checked']

retrieval -> graph {'ids': ['ETR/2004/page_213.pdf:d81f966dc19b::text_6', 'ETR/2008/page_377.pdf:26c3c754ec34::text_6', 'ETR/2011/page_341.pdf:ce32d713fc20::text_2']}
graph -> writer {'matched': 3}


writer -> verifier {'answer': 'unknown'}
verifier -> coordinator {'citation_ids_valid': True, 'semantic_support_checked': False, 'status': 'generated_unverified'}


,agent,evidence,graph,answer,status
0,RetrievalAgent,3,0,,pending
1,GraphAgent,3,3,,pending
2,WriterAgent,3,3,unknown,pending
3,VerifierAgent,3,3,unknown,generated_unverified


## 4. Kiểm tra các đường lỗi

    `ask` chặn domain sai và câu hỏi rỗng. Khi report không tồn tại, truy hồi
    trả rỗng, Writer không sinh và Verifier trả insufficient_evidence.
    Điểm dừng: mỗi tác tử chạy một lần, không vòng lặp đến khi đồng ý.

In [5]:
# Cell AGENT-05
missing = team.ask('Find revenue','finance',report='__missing_report__')
print('Không có nguồn:',missing.status,missing.answer)
assert missing.status=='insufficient_evidence'
for question, domain in [('', 'finance'), ('test','invalid')]:
    try:
        team.ask(question,domain)
    except ValueError as exc:
        print('Đầu vào bị chặn:',exc)
    else:
        raise AssertionError('Đầu vào sai không bị chặn')

Không có nguồn: insufficient_evidence Insufficient evidence
Đầu vào bị chặn: Empty question
Đầu vào bị chặn: Unsupported domain


## 5. Vì sao nguồn tồn tại chưa chứng minh câu trả lời đúng?

    Chạy một câu hỏi đếm qua bộ sinh rồi đối chiếu công cụ statistics của
    ToolAgent. Chỉ statistics đọc số từ bảng thật; không dùng câu trả lời
    sai của generator như dữ kiện. Đây là phép thử giới hạn Verifier.

In [6]:
# Cell AGENT-06
question = 'How many encounters are in the dataset?'
generated = team.ask(question,'diabetes')
computed = agent().ask(question,'diabetes',task='statistics')
print('Sinh:',generated.answer,generated.status)
print('Thống kê:',computed['answer'],computed['status'])
print('Nguồn id hợp lệ:',
      generated.messages[-1].payload['citation_ids_valid'])
payload = {'finance_trace':trace,
    'messages':[vars(m) for m in state.messages],
    'generated_count':generated.answer,
    'computed_count':computed['answer'],
    'missing_status':missing.status}
(OUT/'agents_demo.json').write_text(
    json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf8')

Sinh: 2 generated_unverified
Thống kê: 101766 computed
Nguồn id hợp lệ: True


1440

## Câu hỏi trình bày

    Hãy giải thích riêng State.send, RetrievalAgent.act, GraphAgent.act,
    WriterAgent.act, VerifierAgent.act và MultiAgentTeam.ask. Chỉ ra trường
    State nào bị đổi ở từng bước, Message gửi cho ai và điều kiện dừng/lỗi.
    Tại sao không được đổi generated_unverified thành verified chỉ vì id hợp lệ?